# 🛡️ ScamShield-VN — Kaggle Notebook 1: Tier 1 Foundational Papers & Edge ONNX Benchmark (Zero-Trust)
**Môi trường khuyến nghị trên Kaggle:** `Accelerator: GPU T4 x2` (hoặc `GPU P100`) | `Internet: ON` | **Chi phí API:** `$0.00`

Notebook này thực thi độc lập và xuất toàn bộ dữ liệu thô (`TP, FP, TN, FN` số nguyên từng seed, xác suất dự đoán từng mẫu, thời gian trễ CPU từng mẫu) cho:
1. **Kiểm định SHA-256 & Chống rò rỉ dữ liệu (Anti-Leakage Guard):** Khóa cứng 3 tệp dữ liệu gốc (`N=2,664`, Frozen Test `N=267`, Hard Challenge `N=117`).
2. **Baselines Học máy truyền thống (Theme 1 - Saeed 2023 / Xie 2025):** `TF-IDF + Linear SVM` và `Multinomial Naive Bayes` trên 5 seeds.
3. **Paper Nền tảng số 2 (HC Phap, 2026 - `M041`):** Tái lập kiến trúc `BiGRU + MaskedPool + Cost-Sensitive WBCE (α=5.0)` trên 5 seeds độc lập.
4. **Paper Nền tảng số 1 (Nguyen et al., EMNLP 2023 - `ViSoBERT`) & Các Baselines Transformer (Theme 2):**
   - `visobert_wbce_a5` (**Mô hình đề xuất Tier 1:** `uitnlp/visobert` + Cost-Sensitive WBCE $\alpha=5.0$)
   - `visobert_vanilla_a1` (**Paper nền tảng 1 gốc:** `uitnlp/visobert` + Standard Unweighted BCE $\alpha=1.0$ — cô lập hoàn toàn đóng góp của hàm loss WBCE)
   - `phobert_base_v2` (`vinai/phobert-base-v2` + tách từ `pyvi` + Standard BCE)
   - `fpt_vibert_base` (`FPTAI/vibert-base-cased` + Standard BCE)
   - `phobert_large` (`vinai/phobert-large` + tách từ `pyvi` + Standard BCE)
5. **Xuất ONNX FP32 & INT8 Dynamic Quantization + Đo CPU Latency + Kiểm định McNemar:** Xuất xác suất cho cả `frozen_test_267.csv` và `master_test_suite.jsonl` (`N=384` tổng cộng) để chuyển sang Notebook 2.

In [ ]:
# ==============================================================================
# CELL 1: CÀI ĐẶT THƯ VIỆN & KIỂM ĐỊNH SHA-256 CỨNG (CHỐNG RÒ RỈ DỮ LIỆU 100%)
# ==============================================================================
!pip install -q -U transformers accelerate datasets sentencepiece protobuf onnx onnxruntime scipy scikit-learn pyvi

import os, glob, hashlib, time, random, json, re, unicodedata, shutil, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support

# Tự động dò đường dẫn file trong /kaggle/input
def find_input_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    assert len(matches) > 0, f"❌ Không tìm thấy file {filename} trong /kaggle/input! Hãy gắn Kaggle Dataset scamshield-vn-locked-data."
    return matches[0]

MERGED_PATH = find_input_file("vietnamese_sms_merged_all.csv")
FROZEN_TEST_PATH = find_input_file("frozen_test_267.csv")
SUITE_PATH = find_input_file("master_test_suite.jsonl")

EXPECTED_MERGED_SHA256 = "a468779de45f9d77b7187c828c6d5ac14a801c10b4022853bbc14845573d76e5"
EXPECTED_TEST_SHA256   = "42e18761b12eb299d3ab1c450b29200a38c6dc0334c461c7912254987a85e1e6"
EXPECTED_SUITE_SHA256  = "cf882c44c6a8018dd52bdb3b23405700152e3874a8829f7c026530fd4905021a"

def verify_sha256(filepath, expected_hash):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(8192):
            h.update(chunk)
    actual = h.hexdigest()
    assert actual == expected_hash, f"❌ LỖI SHA-256 tại {filepath}!\nKỳ vọng: {expected_hash}\nThực tế: {actual}"
    print(f"✅ Xác thực SHA-256 khớp tuyệt đối: {os.path.basename(filepath)} ({actual[:16]}...)")

verify_sha256(MERGED_PATH, EXPECTED_MERGED_SHA256)
verify_sha256(FROZEN_TEST_PATH, EXPECTED_TEST_SHA256)
verify_sha256(SUITE_PATH, EXPECTED_SUITE_SHA256)

def clean_text(text):
    if not isinstance(text, str): return ""
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r'[\u200b\u200c\u200d\uFEFF]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

df_all = pd.read_csv(MERGED_PATH)
df_test_frozen = pd.read_csv(FROZEN_TEST_PATH)
with open(SUITE_PATH, "r", encoding="utf-8") as f:
    suite_records = [json.loads(line) for line in f if line.strip()]
df_suite = pd.DataFrame({
    "message_id": [r["sample_id"] for r in suite_records],
    "message": [r["raw_message"] for r in suite_records],
    "label": [1 if r["gold_label"] == "SCAM" else 0 for r in suite_records]
})

assert len(df_all) == 2664, f"Expected 2664 rows, got {len(df_all)}"
assert len(df_test_frozen) == 267, f"Expected 267 test rows, got {len(df_test_frozen)}"
assert len(df_suite) == 117, f"Expected 117 challenge rows, got {len(df_suite)}"

# Cô lập 267 mẫu Frozen Test khỏi tập Phát triển (Train + Val = 2,397 mẫu)
frozen_indices = set(df_test_frozen["original_index"].tolist())
df_dev = df_all.loc[~df_all.index.isin(frozen_indices)].copy().reset_index(drop=True)
df_test = df_test_frozen.copy().reset_index(drop=True)

df_dev["cleaned_message"] = df_dev["message"].apply(clean_text)
df_test["cleaned_message"] = df_test["message"].apply(clean_text)
df_suite["cleaned_message"] = df_suite["message"].apply(clean_text)

# Kiểm tra 0% trùng lặp chuỗi tuyệt đối (Exact String Disjointness) giữa Dev và Frozen Test
overlap = set(df_dev["cleaned_message"]).intersection(set(df_test["cleaned_message"]))
assert len(overlap) == 0, f"❌ Phát hiện {len(overlap)} mẫu rò rỉ giữa Dev và Frozen Test!"
print(f"✅ Phân tách chuẩn xác: Dev (Train+Val) = {len(df_dev)} | Frozen Test = {len(df_test)} (192 Ham, 75 Scam) | Hard Suite = {len(df_suite)} (19 Ham, 98 Scam)")

SEEDS = [42, 100, 123, 999, 2026]
OUT_DIR = "/kaggle/working/results_tier1"
os.makedirs(OUT_DIR, exist_ok=True)

def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    acc = accuracy_score(y_true, y_pred) * 100.0
    p_scam, r_scam, f1_scam, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    _, _, macro_f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    fpr = (fp / (fp + tn)) * 100.0 if (fp + tn) > 0 else 0.0
    return {
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "Accuracy": round(acc, 2), "Precision": round(p_scam * 100.0, 2),
        "Scam_Recall": round(r_scam * 100.0, 2), "Scam_F1": round(f1_scam * 100.0, 2),
        "Macro_F1": round(macro_f1 * 100.0, 2), "FPR": round(fpr, 2)
    }

In [ ]:
# ==============================================================================
# CELL 2: BASELINES HỌC MÁY TRUYỀN THỐNG (TF-IDF + SVM & MULTINOMIAL NAIVE BAYES)
# ==============================================================================
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB

classical_val_records = []
frozen_test_predictions = {
    "message_id": df_test["message_id"].values,
    "gold_label": df_test["label"].values
}

for seed in SEEDS:
    tr_df, va_df = train_test_split(df_dev, test_size=532, random_state=seed, stratify=df_dev["label"])
    
    # 1. TF-IDF + Linear SVM
    tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=15000)
    X_tr_tf = tfidf.fit_transform(tr_df["cleaned_message"])
    X_va_tf = tfidf.transform(va_df["cleaned_message"])
    svm_clf = CalibratedClassifierCV(LinearSVC(random_state=seed), cv=3)
    svm_clf.fit(X_tr_tf, tr_df["label"].values)
    va_probs_svm = svm_clf.predict_proba(X_va_tf)[:, 1]
    res_svm = evaluate_predictions(va_df["label"].values, va_probs_svm, threshold=0.5)
    res_svm["model"] = "tfidf_linear_svm"
    res_svm["seed"] = seed
    classical_val_records.append(res_svm)
    
    # 2. Multinomial Naive Bayes
    cvec = CountVectorizer(ngram_range=(1, 2), max_features=15000)
    X_tr_cv = cvec.fit_transform(tr_df["cleaned_message"])
    X_va_cv = cvec.transform(va_df["cleaned_message"])
    nb_clf = MultinomialNB(alpha=1.0)
    nb_clf.fit(X_tr_cv, tr_df["label"].values)
    va_probs_nb = nb_clf.predict_proba(X_va_cv)[:, 1]
    res_nb = evaluate_predictions(va_df["label"].values, va_probs_nb, threshold=0.5)
    res_nb["model"] = "multinomial_nb"
    res_nb["seed"] = seed
    classical_val_records.append(res_nb)
    
    if seed == 42:
        frozen_test_predictions["tfidf_svm_prob"] = svm_clf.predict_proba(tfidf.transform(df_test["cleaned_message"]))[:, 1]
        frozen_test_predictions["multinomial_nb_prob"] = nb_clf.predict_proba(cvec.transform(df_test["cleaned_message"]))[:, 1]

df_classical = pd.DataFrame(classical_val_records)
df_classical.to_csv(f"{OUT_DIR}/classical_baselines_5seeds_val.csv", index=False)
print("📊 KẾT QUẢ 5 SEEDS CỦA BASELINES HỌC MÁY TRUYỀN THỐNG (THEME 1):")
print(df_classical.groupby("model")[["Precision", "Scam_Recall", "Scam_F1", "Macro_F1"]].agg(["mean", "std"]))

In [ ]:
# ==============================================================================
# CELL 3: PAPER NỀN TẢNG SỐ 2 (HC PHAP 2026) - BiGRU + MaskedPool + WBCE (α=5.0)
# ==============================================================================
from collections import Counter
from torch.utils.data import DataLoader

class SimpleVocabTokenizer:
    def __init__(self, max_vocab=15000, max_len=128):
        self.max_vocab = max_vocab
        self.max_len = max_len
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}

    def fit(self, texts):
        counter = Counter()
        for t in texts:
            counter.update(t.lower().split())
        for w, _ in counter.most_common(self.max_vocab - 2):
            self.word2idx[w] = len(self.word2idx)

    def encode_batch(self, texts):
        ids_list, mask_list = [], []
        for t in texts:
            tokens = t.lower().split()[:self.max_len]
            ids = [self.word2idx.get(w, 1) for w in tokens]
            mask = [1] * len(ids)
            pad_len = self.max_len - len(ids)
            ids += [0] * pad_len
            mask += [0] * pad_len
            ids_list.append(ids)
            mask_list.append(mask)
        return torch.tensor(ids_list, dtype=torch.long), torch.tensor(mask_list, dtype=torch.float32)

class BiGRUMaskedPoolClassifier(nn.Module):
    """Kiến trúc BiGRU + Masked Global Pooling theo HC Phap (2026)"""
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bigru = nn.GRU(embed_dim, hidden_dim, num_layers=2, bidirectional=True, batch_first=True, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 4, 1)

    def forward(self, input_ids, attention_mask):
        emb = self.dropout(self.embedding(input_ids))
        out, _ = self.bigru(emb)
        mask = attention_mask.unsqueeze(-1)
        sum_pooled = torch.sum(out * mask, dim=1)
        lengths = torch.clamp(mask.sum(dim=1), min=1e-9)
        mean_pooled = sum_pooled / lengths
        masked_out = out.masked_fill(mask == 0, -1e9)
        max_pooled = torch.max(masked_out, dim=1)[0]
        feat = self.dropout(torch.cat([mean_pooled, max_pooled], dim=-1))
        return self.fc(feat).squeeze(-1)

bigru_val_records = []
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for seed in SEEDS:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tr_df, va_df = train_test_split(df_dev, test_size=532, random_state=seed, stratify=df_dev["label"])
    
    vocab_tok = SimpleVocabTokenizer(max_vocab=15000, max_len=128)
    vocab_tok.fit(tr_df["cleaned_message"].tolist())
    
    X_tr, M_tr = vocab_tok.encode_batch(tr_df["cleaned_message"].tolist())
    y_tr = torch.tensor(tr_df["label"].values, dtype=torch.float32)
    X_va, M_va = vocab_tok.encode_batch(va_df["cleaned_message"].tolist())
    y_va = va_df["label"].values
    
    train_loader = DataLoader(list(zip(X_tr, M_tr, y_tr)), batch_size=32, shuffle=True)
    model_bigru = BiGRUMaskedPoolClassifier(len(vocab_tok.word2idx)).to(device)
    optimizer = torch.optim.AdamW(model_bigru.parameters(), lr=1e-3, weight_decay=0.01)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([5.0], device=device))
    
    best_macro_f1 = -1.0
    best_va_probs = None
    best_state = None
    
    for epoch in range(8):
        model_bigru.train()
        for bx, bm, by in train_loader:
            bx, bm, by = bx.to(device), bm.to(device), by.to(device)
            optimizer.zero_grad()
            logits = model_bigru(bx, bm)
            loss = criterion(logits, by)
            loss.backward()
            optimizer.step()
            
        model_bigru.eval()
        with torch.no_grad():
            va_logits = model_bigru(X_va.to(device), M_va.to(device)).cpu().numpy()
            va_probs = 1.0 / (1.0 + np.exp(-va_logits))
            metrics_va = evaluate_predictions(y_va, va_probs, threshold=0.5)
            if metrics_va["Macro_F1"] > best_macro_f1:
                best_macro_f1 = metrics_va["Macro_F1"]
                best_va_probs = va_probs
                best_state = {k: v.cpu() for k, v in model_bigru.state_dict().items()}
                
    res = evaluate_predictions(y_va, best_va_probs, threshold=0.5)
    res["model"] = "bigru_maskedpool_wbce_a5"
    res["seed"] = seed
    bigru_val_records.append(res)
    
    if seed == 42:
        model_bigru.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        model_bigru.eval()
        X_te, M_te = vocab_tok.encode_batch(df_test["cleaned_message"].tolist())
        with torch.no_grad():
            te_logits = model_bigru(X_te.to(device), M_te.to(device)).cpu().numpy()
            frozen_test_predictions["bigru_wbce_prob"] = 1.0 / (1.0 + np.exp(-te_logits))
        torch.save(best_state, f"{OUT_DIR}/bigru_wbce_seed42.pt")

df_bigru_val = pd.DataFrame(bigru_val_records)
df_bigru_val.to_csv(f"{OUT_DIR}/bigru_wbce_5seeds_val.csv", index=False)
print("📊 KẾT QUẢ 5 SEEDS CỦA PAPER NỀN TẢNG 2 (BiGRU + MaskedPool + WBCE α=5.0):")
print(df_bigru_val.to_string(index=False))

In [ ]:
# ==============================================================================
# CELL 4: PAPER NỀN TẢNG SỐ 1 (VISOBERT EMNLP 2023) & CÁC BASELINE TRANSFORMERS
# ==============================================================================
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from pyvi import ViTokenizer

MODEL_CONFIGS = [
    {"key": "visobert_wbce_a5",    "hf_repo": "uitnlp/visobert",         "pos_weight": 5.0, "use_pyvi": False, "batch_size": 32},
    {"key": "visobert_vanilla_a1", "hf_repo": "uitnlp/visobert",         "pos_weight": 1.0, "use_pyvi": False, "batch_size": 32},
    {"key": "phobert_base_v2",     "hf_repo": "vinai/phobert-base-v2",   "pos_weight": 1.0, "use_pyvi": True,  "batch_size": 32},
    {"key": "fpt_vibert_base",     "hf_repo": "FPTAI/vibert-base-cased", "pos_weight": 1.0, "use_pyvi": False, "batch_size": 32},
    {"key": "phobert_large",       "hf_repo": "vinai/phobert-large",     "pos_weight": 1.0, "use_pyvi": True,  "batch_size": 16},
]

class CostSensitiveBCETrainer(Trainer):
    def __init__(self, pos_weight_val=1.0, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight_val = pos_weight_val

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels").float()
        outputs = model(**inputs)
        logits = outputs.get("logits").squeeze(-1)
        pw = torch.tensor([self.pos_weight_val], device=logits.device, dtype=logits.dtype)
        loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

all_val_summary = []
eval_key = "eval_strategy" if hasattr(TrainingArguments, "eval_strategy") else "evaluation_strategy"

for cfg in MODEL_CONFIGS:
    m_key = cfg["key"]
    hf_repo = cfg["hf_repo"]
    pos_w = cfg["pos_weight"]
    use_pyvi = cfg["use_pyvi"]
    bs = cfg["batch_size"]
    
    print(f"\n==================================================================")
    print(f"🚀 ĐANG CHẠY MÔ HÌNH: {m_key} ({hf_repo} | α={pos_w} | pyvi={use_pyvi})")
    print(f"==================================================================")
    
    tokenizer = AutoTokenizer.from_pretrained(hf_repo)
    
    def prepare_texts(series):
        if use_pyvi:
            return [ViTokenizer.tokenize(t) for t in series.tolist()]
        return series.tolist()

    def make_hf_dataset(df_sub):
        ds = Dataset.from_dict({
            "text": prepare_texts(df_sub["cleaned_message"]),
            "labels": df_sub["label"].astype(int).tolist()
        })
        return ds.map(lambda ex: tokenizer(ex["text"], padding="max_length", truncation=True, max_length=128), batched=True)

    test_ds = make_hf_dataset(df_test)
    
    for seed in SEEDS:
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        
        tr_df, va_df = train_test_split(df_dev, test_size=532, random_state=seed, stratify=df_dev["label"])
        train_ds = make_hf_dataset(tr_df)
        val_ds = make_hf_dataset(va_df)
        
        # KHỞI TẠO MỚI 100% TỪ BASE MODEL GỐC CHO TỪNG SEED (KHÔNG NỐI TIẾP CHECKPOINT)
        model = AutoModelForSequenceClassification.from_pretrained(hf_repo, num_labels=1)
        
        def compute_val_metrics(eval_pred):
            logits, labels = eval_pred
            probs = 1.0 / (1.0 + np.exp(-logits.squeeze(-1)))
            preds = (probs >= 0.5).astype(int)
            _, _, macro_f1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
            return {"macro_f1": macro_f1}

        t_args_dict = {
            "output_dir": f"/kaggle/working/tmp_{m_key}_s{seed}",
            eval_key: "epoch",
            "save_strategy": "epoch",
            "save_total_limit": 1,
            "learning_rate": 2e-5,
            "weight_decay": 0.01,
            "warmup_ratio": 0.1,
            "per_device_train_batch_size": bs,
            "per_device_eval_batch_size": 64,
            "num_train_epochs": 5,
            "fp16": torch.cuda.is_available(),
            "load_best_model_at_end": True,
            "metric_for_best_model": "macro_f1",
            "greater_is_better": True,
            "seed": seed,
            "report_to": "none",
            "logging_strategy": "no"
        }
        training_args = TrainingArguments(**t_args_dict)
        
        trainer = CostSensitiveBCETrainer(
            pos_weight_val=pos_w,
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            compute_metrics=compute_val_metrics
        )
        trainer.train()
        
        val_out = trainer.predict(val_ds)
        val_probs = 1.0 / (1.0 + np.exp(-val_out.predictions.squeeze(-1)))
        val_metrics = evaluate_predictions(va_df["label"].values, val_probs, threshold=0.5)
        val_metrics["model"] = m_key
        val_metrics["seed"] = seed
        all_val_summary.append(val_metrics)
        print(f"  [Seed {seed}] Val TP={val_metrics['TP']} FP={val_metrics['FP']} TN={val_metrics['TN']} FN={val_metrics['FN']} | ScamRecall={val_metrics['Scam_Recall']}% | MacroF1={val_metrics['Macro_F1']}%")
        
        if seed == 42:
            test_out = trainer.predict(test_ds)
            test_probs = 1.0 / (1.0 + np.exp(-test_out.predictions.squeeze(-1)))
            frozen_test_predictions[f"{m_key}_prob"] = test_probs
            if m_key == "visobert_wbce_a5":
                save_dir = f"{OUT_DIR}/visobert_wbce_seed42_best"
                model.save_pretrained(save_dir)
                tokenizer.save_pretrained(save_dir)
                
        del trainer, model
        torch.cuda.empty_cache()
        gc.collect()
        shutil.rmtree(f"/kaggle/working/tmp_{m_key}_s{seed}", ignore_errors=True)

df_val_all = pd.DataFrame(all_val_summary)
df_val_all.to_csv(f"{OUT_DIR}/all_transformers_5seeds_val_metrics.csv", index=False)
pd.DataFrame(frozen_test_predictions).to_csv(f"{OUT_DIR}/tier1_frozen_test_267_probs.csv", index=False)
print("\n✅ ĐÃ LƯU XONG TOÀN BỘ KẾT QUẢ 5 SEEDS VÀ XÁC SUẤT FROZEN TEST (N=267)!")

In [ ]:
# ==============================================================================
# CELL 5: LƯỢNG TỬ HÓA ONNX INT8, ĐO CPU LATENCY (N=267 + N=117) & MCNEMAR TEST
# ==============================================================================
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
from scipy.stats import binomtest

best_dir = f"{OUT_DIR}/visobert_wbce_seed42_best"
onnx_fp32_path = f"{OUT_DIR}/visobert_tier1_fp32.onnx"
onnx_int8_path = f"{OUT_DIR}/visobert_tier1_int8.onnx"

tok_visobert = AutoTokenizer.from_pretrained(best_dir)
mod_visobert = AutoModelForSequenceClassification.from_pretrained(best_dir).eval().cpu()

dummy = tok_visobert("Vietcombank thong bao cap nhat sinh trac hoc tai vcb-online.top", return_tensors="pt", padding="max_length", max_length=128, truncation=True)
torch.onnx.export(
    mod_visobert,
    (dummy["input_ids"], dummy["attention_mask"]),
    onnx_fp32_path,
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    dynamic_axes={"input_ids": {0: "batch"}, "attention_mask": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=14
)

quantize_dynamic(onnx_fp32_path, onnx_int8_path, weight_type=QuantType.QInt8)

fp32_mb = os.path.getsize(onnx_fp32_path) / (1024 * 1024)
int8_mb = os.path.getsize(onnx_int8_path) / (1024 * 1024)
print(f"📦 Kích thước ONNX FP32: {fp32_mb:.2f} MB | ONNX INT8: {int8_mb:.2f} MB (Giảm {(1.0 - int8_mb/fp32_mb)*100.0:.1f}%)")

sess_options = ort.SessionOptions()
sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1
ort_sess = ort.InferenceSession(onnx_int8_path, sess_options, providers=["CPUExecutionProvider"])

in_feed = {"input_ids": dummy["input_ids"].numpy(), "attention_mask": dummy["attention_mask"].numpy()}
for _ in range(50):
    _ = ort_sess.run(None, in_feed)

def run_onnx_inference(texts):
    lats, probs = [], []
    for msg in texts:
        enc = tok_visobert(msg, return_tensors="np", padding="max_length", max_length=128, truncation=True)
        feed = {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}
        t0 = time.perf_counter()
        logit = ort_sess.run(None, feed)[0].squeeze()
        t1 = time.perf_counter()
        lats.append((t1 - t0) * 1000.0)
        probs.append(float(1.0 / (1.0 + np.exp(-logit))))
    return probs, lats

test_probs_int8, test_lats_int8 = run_onnx_inference(df_test["cleaned_message"].tolist())
suite_probs_int8, suite_lats_int8 = run_onnx_inference(df_suite["cleaned_message"].tolist())

df_probs = pd.read_csv(f"{OUT_DIR}/tier1_frozen_test_267_probs.csv")
df_probs["visobert_int8_onnx_prob"] = test_probs_int8
df_probs["visobert_int8_latency_ms"] = test_lats_int8
df_probs.to_csv(f"{OUT_DIR}/tier1_frozen_test_267_probs.csv", index=False)

df_suite_out = pd.DataFrame({
    "message_id": df_suite["message_id"].values,
    "gold_label": df_suite["label"].values,
    "visobert_int8_onnx_prob": suite_probs_int8,
    "visobert_int8_latency_ms": suite_lats_int8
})
df_suite_out.to_csv(f"{OUT_DIR}/tier1_challenge_117_probs.csv", index=False)

print(f"⚡ ONNX INT8 CPU Latency (N=267): Median = {np.median(test_lats_int8):.2f} ms | Mean = {np.mean(test_lats_int8):.2f} ± {np.std(test_lats_int8):.2f} ms | P95 = {np.percentile(test_lats_int8, 95):.2f} ms")

def mcnemar_paired_test(y_true, pred_a, pred_b, name_a, name_b):
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    b = int(np.sum(correct_a & (~correct_b)))
    c = int(np.sum((~correct_a) & correct_b))
    chi2 = ((abs(b - c) - 1.0) ** 2) / (b + c) if (b + c) > 0 else 0.0
    pval = binomtest(min(b, c), n=b+c, p=0.5, alternative="two-sided").pvalue if (b + c) > 0 else 1.0
    odds_ratio = (b / c) if c > 0 else float("inf")
    return {"Comparison": f"{name_a} vs. {name_b}", "b (A_ok_B_err)": b, "c (A_err_B_ok)": c, "chi2_edwards": round(chi2, 4), "p_value": round(pval, 6), "odds_ratio": round(odds_ratio, 2)}

y_gold = df_probs["gold_label"].values
pred_ours = (df_probs["visobert_int8_onnx_prob"].values >= 0.90).astype(int)

mcnemar_rows = []
for col, label_name in [
    ("visobert_vanilla_a1_prob", "Vanilla ViSoBERT (α=1.0)"),
    ("bigru_wbce_prob", "BiGRU+MaskedPool+WBCE (HC Phap 2026)"),
    ("phobert_base_v2_prob", "PhoBERT-base-v2"),
    ("fpt_vibert_base_prob", "FPT ViBERT-base"),
    ("phobert_large_prob", "PhoBERT-large"),
    ("tfidf_svm_prob", "TF-IDF + Linear SVM"),
    ("multinomial_nb_prob", "Multinomial Naive Bayes")
]:
    if col in df_probs.columns:
        pred_base = (df_probs[col].values >= 0.50).astype(int)
        mcnemar_rows.append(mcnemar_paired_test(y_gold, pred_ours, pred_base, "ViSoBERT-INT8(ours)", label_name))

df_mcnemar = pd.DataFrame(mcnemar_rows)
df_mcnemar.to_csv(f"{OUT_DIR}/mcnemar_tier1_tests.csv", index=False)
print("\n📊 BẢNG KIỂM ĐỊNH THỐNG KÊ MCNEMAR TRÊN FROZEN TEST (N=267):")
print(df_mcnemar.to_string(index=False))

shutil.make_archive("/kaggle/working/scamshield_tier1_all_artifacts", "zip", OUT_DIR)
print("\n📦 Đã đóng gói toàn bộ kết quả tại: /kaggle/working/scamshield_tier1_all_artifacts.zip")